# MLP - AESO Price Spike Forecasting

This notebook builds a simple feed-forward neural network to predict `spike_lead_2` in Alberta hourly electricity prices.
It follows the same structure as the LSTM notebook, but because the MLP is not sequential, the lag features are included directly in the input table.


In [1]:
# 1. Imports and config
import json
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import fbeta_score, f1_score, precision_score, recall_score, roc_auc_score, average_precision_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# These paths are relative to this notebook's folder: JorgeFolder/models/mlp/
SOURCE_DATA = Path("../../../Data/CSVs/aeso_merged_2020_2025.csv")
OUTPUT_DIR = Path("random_search_run")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

if not SOURCE_DATA.exists():
    raise FileNotFoundError("Could not resolve ../../../Data/CSVs/aeso_merged_2020_2025.csv from the notebook folder.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "spike_lead_2"
TRAIN_END_EXCLUSIVE = pd.Timestamp("2023-11-06 00:00:00")
VAL_END_EXCLUSIVE = pd.Timestamp("2024-12-12 00:00:00")
TEST_START = VAL_END_EXCLUSIVE
N_CV_SPLITS = 5
MAX_EPOCHS = 10
PATIENCE = 3

# We keep 0.50 only as a quick reference point during training.
# Final candidate ranking and final evaluation use threshold tuning on validation.
THRESHOLD = 0.5
THRESHOLD_GRID = np.linspace(0.05, 0.99, 95)

RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Source data  : {SOURCE_DATA}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Device       : {DEVICE}")


Source data  : ..\..\..\Data\CSVs\aeso_merged_2020_2025.csv
Output dir   : random_search_run
Device       : cpu


In [2]:
# 2. Feature definitions
# We keep the same core system features as the LSTM notebook.
# For the MLP, we also include the lag columns directly because the model does not have memory.

BASE_FEATURES = [
    "ACTUAL_POOL_PRICE",
    "ACTUAL_AIL",
    "gas_total", "wind_total", "solar_total", "coal_total", "hydro_total",
    "net_export", "IMPORT_BC", "IMPORT_MT", "IMPORT_SK",
    "net_load", "net_load_3h_change",
    "reserve_margin", "resilience_buffer",
    "renewables_share", "dispatchable_ratio", "gas_ratio",
    "renewable_generation", "dispatchable_generation",
    "sin_hour", "cos_hour",
    "sin_dow", "cos_dow",
    "sin_year_1", "cos_year_1",
    "sin_hour_2", "cos_hour_2",
    "sin_year_2", "cos_year_2",
    "is_weekend", "is_stampede",
]

SPIKE_LAG_FEATURES = [f"spike_lag_{i}" for i in range(1, 25)]
PRICE_LAG_FEATURES = ["price_lag_1h", "price_lag_6h", "price_lag_24h"]

FEATURES = BASE_FEATURES + SPIKE_LAG_FEATURES + PRICE_LAG_FEATURES

NO_SCALE = [
    "sin_hour", "cos_hour", "sin_dow", "cos_dow",
    "sin_year_1", "cos_year_1", "sin_hour_2", "cos_hour_2", "sin_year_2", "cos_year_2",
    "is_weekend", "is_stampede",
] + SPIKE_LAG_FEATURES

CONTINUOUS = [c for c in FEATURES if c not in NO_SCALE]

BASE_REQUIRED_COLUMNS = [
    "datetime", "ACTUAL_POOL_PRICE", "ACTUAL_AIL",
    "gas_total", "wind_total", "solar_total", "coal_total", "hydro_total",
    "net_export", "IMPORT_BC", "IMPORT_MT", "IMPORT_SK",
    "net_load", "net_load_3h_change", "reserve_margin", "resilience_buffer",
    "renewables_share", "dispatchable_ratio", "gas_ratio",
    "renewable_generation", "dispatchable_generation",
] + SPIKE_LAG_FEATURES + PRICE_LAG_FEATURES

print(f"Feature count: {len(FEATURES)}")


Feature count: 59


In [3]:
# 3. Load and clean
df = pd.read_csv(SOURCE_DATA, parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

missing_required = [c for c in BASE_REQUIRED_COLUMNS + [TARGET] if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

print(f"Raw rows      : {len(df):,}")
print(f"Raw columns   : {len(df.columns):,}")


Raw rows      : 48,887
Raw columns   : 108


In [4]:
# 4. Feature engineering
# The target spike_lead_2 and the lag columns already exist in the merged CSV.
# Here we only add the calendar features and event flags used by the model.

if "is_weekend" not in df.columns:
    df["is_weekend"] = (df["datetime"].dt.dayofweek >= 5).astype(int)

if "is_stampede" not in df.columns:
    df["is_stampede"] = 0
    for start, end in [
        ("2021-07-09", "2021-07-18"), ("2022-07-08", "2022-07-17"),
        ("2023-07-07", "2023-07-16"), ("2024-07-05", "2024-07-14"),
        ("2025-07-04", "2025-07-13"),
    ]:
        df.loc[df["datetime"].between(start, end), "is_stampede"] = 1

doy = df["datetime"].dt.day_of_year.astype(float)

df["sin_hour"] = np.sin(2 * np.pi * df["datetime"].dt.hour / 24)
df["cos_hour"] = np.cos(2 * np.pi * df["datetime"].dt.hour / 24)
df["sin_dow"] = np.sin(2 * np.pi * df["datetime"].dt.dayofweek / 7)
df["cos_dow"] = np.cos(2 * np.pi * df["datetime"].dt.dayofweek / 7)
df["sin_year_1"] = np.sin(2 * np.pi * doy / 365)
df["cos_year_1"] = np.cos(2 * np.pi * doy / 365)

df["sin_hour_2"] = np.sin(4 * np.pi * df["datetime"].dt.hour / 24)
df["cos_hour_2"] = np.cos(4 * np.pi * df["datetime"].dt.hour / 24)
df["sin_year_2"] = np.sin(4 * np.pi * doy / 365)
df["cos_year_2"] = np.cos(4 * np.pi * doy / 365)

df = df.dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
df[TARGET] = df[TARGET].astype(int)

print(f"Modeling rows : {len(df):,}")
print(f"Spike rate    : {df[TARGET].mean():.4f}")


Modeling rows : 48,887


Spike rate    : 0.0994


In [5]:
# 5. Train / validation / test split
train = df[df["datetime"] < TRAIN_END_EXCLUSIVE].reset_index(drop=True)
validation = df[(df["datetime"] >= TRAIN_END_EXCLUSIVE) & (df["datetime"] < VAL_END_EXCLUSIVE)].reset_index(drop=True)
test = df[df["datetime"] >= TEST_START].reset_index(drop=True)

print(f"Train rows      : {len(train):,}")
print(f"Validation rows : {len(validation):,}")
print(f"Test rows       : {len(test):,}")


Train rows      : 33,696
Validation rows : 9,648
Test rows       : 5,543


In [6]:
# 6. Tensor datasets and MLP model
# The MLP sees one row at a time, so we can feed the tabular feature matrix directly.

class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()
        layers = []
        current_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(current_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            current_dim = hidden_dim
        layers.append(nn.Linear(current_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)

print(f"Input dimension: {len(FEATURES)}")


Input dimension: 59


## Why F1 score?
With spikes representing roughly 9% of observations, accuracy would be misleading.
F1 is the main selection metric here because it balances precision and recall more evenly.
We still report F2 as a secondary metric because missed spikes still matter economically.


In [7]:
# 7. Small candidate grid
# We keep the grid short so the notebook stays easy to read.
# Each candidate is ranked by its best validation F1 after threshold tuning.

CANDIDATE_GRID = [
    {"hidden_dims": [64], "dropout": 0.10, "lr": 0.0010, "weight_decay": 1e-5, "batch_size": 256},
    {"hidden_dims": [128], "dropout": 0.15, "lr": 0.0010, "weight_decay": 1e-5, "batch_size": 256},
    {"hidden_dims": [128, 64], "dropout": 0.20, "lr": 0.0010, "weight_decay": 1e-5, "batch_size": 256},
    {"hidden_dims": [256, 128], "dropout": 0.20, "lr": 0.0010, "weight_decay": 1e-5, "batch_size": 256},
    {"hidden_dims": [256, 128, 64], "dropout": 0.25, "lr": 0.0008, "weight_decay": 1e-5, "batch_size": 256},
]

pretest = pd.concat([train, validation], ignore_index=True)
y_tv = pretest[TARGET].values.astype(np.float32)
tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
folds = list(tscv.split(pretest))
tr_tune_idx, va_tune_idx = folds[-1]

best_f1 = -1.0
BEST = None
search_results = []

for trial, params in enumerate(CANDIDATE_GRID, start=1):
    scaler = StandardScaler()
    scaler.fit(pretest.iloc[tr_tune_idx][CONTINUOUS])

    X_tv_sc = pretest[FEATURES].copy().astype(float)
    X_tv_sc[CONTINUOUS] = scaler.transform(X_tv_sc[CONTINUOUS])

    X_tr = X_tv_sc.iloc[tr_tune_idx].values.astype(np.float32)
    y_tr = y_tv[tr_tune_idx]
    X_va = X_tv_sc.iloc[va_tune_idx].values.astype(np.float32)
    y_va = y_tv[va_tune_idx]

    tr_loader = DataLoader(
        TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
        batch_size=params["batch_size"],
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    va_loader = DataLoader(
        TensorDataset(torch.tensor(X_va), torch.tensor(y_va)),
        batch_size=params["batch_size"] * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    n_pos = float(y_tr.sum())
    n_neg = float(len(y_tr) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)

    model = MLPClassifier(len(FEATURES), params["hidden_dims"], params["dropout"]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])

    trial_best_f1 = -1.0
    trial_best_f2 = -1.0
    trial_best_threshold = THRESHOLD
    patience_ctr = 0

    for _ in range(MAX_EPOCHS):
        model.train()
        for X_batch, y_batch in tr_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch.to(DEVICE)), y_batch.to(DEVICE))
            loss.backward()
            optimizer.step()

        model.eval()
        probs = []
        with torch.no_grad():
            for X_batch, _ in va_loader:
                probs.append(torch.sigmoid(model(X_batch.to(DEVICE))).cpu().numpy())
        probs = np.concatenate(probs)

        epoch_rows = []
        for threshold in THRESHOLD_GRID:
            preds = (probs >= threshold).astype(int)
            epoch_rows.append(
                {
                    "threshold": float(threshold),
                    "f1": float(f1_score(y_va, preds, zero_division=0)),
                    "f2": float(fbeta_score(y_va, preds, beta=2, zero_division=0)),
                    "precision": float(precision_score(y_va, preds, zero_division=0)),
                }
            )

        epoch_best = pd.DataFrame(epoch_rows).sort_values(
            ["f1", "f2", "precision", "threshold"], ascending=[False, False, False, False]
        ).reset_index(drop=True).iloc[0]

        if float(epoch_best["f1"]) > trial_best_f1:
            trial_best_f1 = float(epoch_best["f1"])
            trial_best_f2 = float(epoch_best["f2"])
            trial_best_threshold = float(epoch_best["threshold"])
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

    search_results.append(
        {
            "trial": trial,
            "hidden_dims": str(params["hidden_dims"]),
            "dropout": params["dropout"],
            "lr": params["lr"],
            "weight_decay": params["weight_decay"],
            "batch_size": params["batch_size"],
            "best_threshold": round(float(trial_best_threshold), 2),
            "best_f1": round(float(trial_best_f1), 4),
            "best_f2_at_best_f1_threshold": round(float(trial_best_f2), 4),
        }
    )
    print(
        f"Trial {trial:2d}/{len(CANDIDATE_GRID)}  "
        f"F1={trial_best_f1:.4f}  F2={trial_best_f2:.4f}  threshold={trial_best_threshold:.2f}  {params}"
    )

    if trial_best_f1 > best_f1:
        best_f1 = trial_best_f1
        BEST = params

search_results_df = pd.DataFrame(search_results).sort_values("best_f1", ascending=False).reset_index(drop=True)
print()
print(f"Best config  : {BEST}")
print(f"Best tune F1 : {best_f1:.4f}")
search_results_df


Trial  1/5  F1=0.5637  F2=0.5511  threshold=0.88  {'hidden_dims': [64], 'dropout': 0.1, 'lr': 0.001, 'weight_decay': 1e-05, 'batch_size': 256}


Trial  2/5  F1=0.5821  F2=0.5481  threshold=0.89  {'hidden_dims': [128], 'dropout': 0.15, 'lr': 0.001, 'weight_decay': 1e-05, 'batch_size': 256}


Trial  3/5  F1=0.5700  F2=0.5339  threshold=0.92  {'hidden_dims': [128, 64], 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 1e-05, 'batch_size': 256}


Trial  4/5  F1=0.5697  F2=0.5665  threshold=0.89  {'hidden_dims': [256, 128], 'dropout': 0.2, 'lr': 0.001, 'weight_decay': 1e-05, 'batch_size': 256}


Trial  5/5  F1=0.5714  F2=0.5521  threshold=0.90  {'hidden_dims': [256, 128, 64], 'dropout': 0.25, 'lr': 0.0008, 'weight_decay': 1e-05, 'batch_size': 256}

Best config  : {'hidden_dims': [128], 'dropout': 0.15, 'lr': 0.001, 'weight_decay': 1e-05, 'batch_size': 256}
Best tune F1 : 0.5821


,trial,hidden_dims,dropout,lr,weight_decay,batch_size,best_threshold,best_f1,best_f2_at_best_f1_threshold
0,2,[128],0.15,0.0010,0.00001,256,0.89,0.5821,0.5481
1,5,"[256, 128, 64]",0.25,0.0008,0.00001,256,0.90,0.5714,0.5521
2,3,"[128, 64]",0.20,0.0010,0.00001,256,0.92,0.5700,0.5339
3,4,"[256, 128]",0.20,0.0010,0.00001,256,0.89,0.5697,0.5665
4,1,[64],0.10,0.0010,0.00001,256,0.88,0.5637,0.5511


In [8]:
# 8. Full TimeSeriesSplit cross-validation with the best configuration
# We keep the same idea here: each fold is judged by the best validation F1 after threshold tuning.
pretest = pd.concat([train, validation], ignore_index=True)
y_tv = pretest[TARGET].values.astype(np.float32)
fold_f1s = []
fold_f2s_at_best_f1_threshold = []
fold_thresholds = []

for fold, (tr_idx, va_idx) in enumerate(folds, 1):
    scaler = StandardScaler()
    scaler.fit(pretest.iloc[tr_idx][CONTINUOUS])

    X_tv_sc = pretest[FEATURES].copy().astype(float)
    X_tv_sc[CONTINUOUS] = scaler.transform(X_tv_sc[CONTINUOUS])

    X_tr = X_tv_sc.iloc[tr_idx].values.astype(np.float32)
    y_tr = y_tv[tr_idx]
    X_va = X_tv_sc.iloc[va_idx].values.astype(np.float32)
    y_va = y_tv[va_idx]

    tr_loader = DataLoader(
        TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
        batch_size=BEST["batch_size"],
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )
    va_loader = DataLoader(
        TensorDataset(torch.tensor(X_va), torch.tensor(y_va)),
        batch_size=BEST["batch_size"] * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    n_pos = float(y_tr.sum())
    n_neg = float(len(y_tr) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)

    model = MLPClassifier(len(FEATURES), BEST["hidden_dims"], BEST["dropout"]).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=BEST["lr"], weight_decay=BEST["weight_decay"])

    best_fold_f1 = -1.0
    best_fold_f2 = -1.0
    best_fold_threshold = THRESHOLD
    patience_ctr = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for X_batch, y_batch in tr_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch.to(DEVICE)), y_batch.to(DEVICE))
            loss.backward()
            optimizer.step()

        model.eval()
        probs = []
        with torch.no_grad():
            for X_batch, _ in va_loader:
                probs.append(torch.sigmoid(model(X_batch.to(DEVICE))).cpu().numpy())
        probs = np.concatenate(probs)

        epoch_rows = []
        for threshold in THRESHOLD_GRID:
            preds = (probs >= threshold).astype(int)
            epoch_rows.append(
                {
                    "threshold": float(threshold),
                    "f1": float(f1_score(y_va, preds, zero_division=0)),
                    "f2": float(fbeta_score(y_va, preds, beta=2, zero_division=0)),
                    "precision": float(precision_score(y_va, preds, zero_division=0)),
                }
            )

        epoch_best = pd.DataFrame(epoch_rows).sort_values(
            ["f1", "f2", "precision", "threshold"], ascending=[False, False, False, False]
        ).reset_index(drop=True).iloc[0]

        if float(epoch_best["f1"]) > best_fold_f1:
            best_fold_f1 = float(epoch_best["f1"])
            best_fold_f2 = float(epoch_best["f2"])
            best_fold_threshold = float(epoch_best["threshold"])
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

    fold_f1s.append(best_fold_f1)
    fold_f2s_at_best_f1_threshold.append(best_fold_f2)
    fold_thresholds.append(best_fold_threshold)
    print(
        f"Fold {fold}/{N_CV_SPLITS}  best F1 = {best_fold_f1:.4f}  "
        f"F2 = {best_fold_f2:.4f}  threshold = {best_fold_threshold:.2f}  (stopped epoch {epoch})"
    )

print()
print(f"Mean CV F1 : {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f}")
print(
    f"Mean CV F2 : {np.mean(fold_f2s_at_best_f1_threshold):.4f} +/- {np.std(fold_f2s_at_best_f1_threshold):.4f}  "
    f"(at the best-F1 thresholds)"
)


Fold 1/5  best F1 = 0.6071  F2 = 0.6051  threshold = 0.89  (stopped epoch 7)


Fold 2/5  best F1 = 0.5450  F2 = 0.5408  threshold = 0.77  (stopped epoch 10)


Fold 3/5  best F1 = 0.6574  F2 = 0.6811  threshold = 0.76  (stopped epoch 9)


Fold 4/5  best F1 = 0.6740  F2 = 0.6730  threshold = 0.89  (stopped epoch 9)


Fold 5/5  best F1 = 0.5782  F2 = 0.5680  threshold = 0.92  (stopped epoch 10)

Mean CV F1 : 0.6123 +/- 0.0481
Mean CV F2 : 0.6136 +/- 0.0558  (at the best-F1 thresholds)


In [9]:
# 9. Final model: train on train, early stop on validation using tuned F1,
# then apply the best validation threshold once to the untouched test set.
full = pd.concat([train, validation, test], ignore_index=True)
y_full = full[TARGET].values.astype(np.float32)

train_idx = np.flatnonzero(full["datetime"] < TRAIN_END_EXCLUSIVE)
val_idx = np.flatnonzero((full["datetime"] >= TRAIN_END_EXCLUSIVE) & (full["datetime"] < VAL_END_EXCLUSIVE))
test_idx = np.flatnonzero(full["datetime"] >= TEST_START)

scaler_final = StandardScaler()
scaler_final.fit(train[CONTINUOUS])

X_full = full[FEATURES].copy().astype(float)
X_full[CONTINUOUS] = scaler_final.transform(X_full[CONTINUOUS])
X_full = X_full.values.astype(np.float32)

X_tr = X_full[train_idx]
y_tr = y_full[train_idx]
X_va = X_full[val_idx]
y_va = y_full[val_idx]
X_te = X_full[test_idx]
y_te = y_full[test_idx]

tr_loader = DataLoader(
    TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr)),
    batch_size=BEST["batch_size"],
    shuffle=True,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
va_loader = DataLoader(
    TensorDataset(torch.tensor(X_va), torch.tensor(y_va)),
    batch_size=BEST["batch_size"] * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
te_loader = DataLoader(
    TensorDataset(torch.tensor(X_te), torch.tensor(y_te)),
    batch_size=BEST["batch_size"] * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

n_pos = float(y_tr.sum())
n_neg = float(len(y_tr) - n_pos)
pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)

final_model = MLPClassifier(len(FEATURES), BEST["hidden_dims"], BEST["dropout"]).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(final_model.parameters(), lr=BEST["lr"], weight_decay=BEST["weight_decay"])

best_val_f1_tuned = -1.0
best_val_f2_tuned = -1.0
best_val_threshold_tuned = THRESHOLD
best_state = None
patience_ctr = 0
training_history = []

for epoch in range(1, MAX_EPOCHS + 1):
    final_model.train()
    batch_losses = []
    for X_batch, y_batch in tr_loader:
        optimizer.zero_grad()
        loss = criterion(final_model(X_batch.to(DEVICE)), y_batch.to(DEVICE))
        loss.backward()
        optimizer.step()
        batch_losses.append(float(loss.item()))

    torch.save(final_model.state_dict(), CHECKPOINT_DIR / "latest_model.pt")

    final_model.eval()
    val_probs = []
    with torch.no_grad():
        for X_batch, _ in va_loader:
            val_probs.append(torch.sigmoid(final_model(X_batch.to(DEVICE))).cpu().numpy())
    val_probs = np.concatenate(val_probs)

    epoch_rows = []
    for threshold in THRESHOLD_GRID:
        preds = (val_probs >= threshold).astype(int)
        epoch_rows.append(
            {
                "threshold": float(threshold),
                "f1": float(f1_score(y_va, preds, zero_division=0)),
                "f2": float(fbeta_score(y_va, preds, beta=2, zero_division=0)),
                "precision": float(precision_score(y_va, preds, zero_division=0)),
                "recall": float(recall_score(y_va, preds, zero_division=0)),
                "predicted_positive_rate": float(preds.mean()),
            }
        )

    epoch_best = pd.DataFrame(epoch_rows).sort_values(
        ["f1", "f2", "precision", "threshold"], ascending=[False, False, False, False]
    ).reset_index(drop=True).iloc[0]

    training_history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(batch_losses)) if batch_losses else np.nan,
            "validation_best_f1": float(epoch_best["f1"]),
            "validation_best_f2": float(epoch_best["f2"]),
            "validation_best_threshold": float(epoch_best["threshold"]),
        }
    )

    if float(epoch_best["f1"]) > best_val_f1_tuned:
        best_val_f1_tuned = float(epoch_best["f1"])
        best_val_f2_tuned = float(epoch_best["f2"])
        best_val_threshold_tuned = float(epoch_best["threshold"])
        best_state = {k: v.cpu().clone() for k, v in final_model.state_dict().items()}
        torch.save(best_state, CHECKPOINT_DIR / "best_model.pt")
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            break

final_model.load_state_dict(best_state)

final_model.eval()
val_probs = []
with torch.no_grad():
    for X_batch, _ in va_loader:
        val_probs.append(torch.sigmoid(final_model(X_batch.to(DEVICE))).cpu().numpy())
val_probs = np.concatenate(val_probs)

threshold_rows = []
for threshold in THRESHOLD_GRID:
    val_preds = (val_probs >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": round(float(threshold), 2),
            "f1": float(f1_score(y_va, val_preds, zero_division=0)),
            "f2": float(fbeta_score(y_va, val_preds, beta=2, zero_division=0)),
            "precision": float(precision_score(y_va, val_preds, zero_division=0)),
            "recall": float(recall_score(y_va, val_preds, zero_division=0)),
            "predicted_positive_rate": float(val_preds.mean()),
        }
    )

threshold_search_df = pd.DataFrame(threshold_rows).sort_values(
    ["f1", "f2", "precision", "threshold"], ascending=[False, False, False, False]
).reset_index(drop=True)

best_threshold_row = threshold_search_df.iloc[0]
SELECTED_THRESHOLD = float(best_threshold_row["threshold"])
validation_threshold_metrics = {
    "threshold": float(best_threshold_row["threshold"]),
    "f1": float(best_threshold_row["f1"]),
    "f2": float(best_threshold_row["f2"]),
    "precision": float(best_threshold_row["precision"]),
    "recall": float(best_threshold_row["recall"]),
    "predicted_positive_rate": float(best_threshold_row["predicted_positive_rate"]),
}

probs = []
with torch.no_grad():
    for X_batch, _ in te_loader:
        probs.append(torch.sigmoid(final_model(X_batch.to(DEVICE))).cpu().numpy())
probs = np.concatenate(probs)
preds = (probs >= SELECTED_THRESHOLD).astype(int)

test_metrics = {
    "f1": float(f1_score(y_te, preds, zero_division=0)),
    "f2": float(fbeta_score(y_te, preds, beta=2, zero_division=0)),
    "precision": float(precision_score(y_te, preds, zero_division=0)),
    "recall": float(recall_score(y_te, preds, zero_division=0)),
    "pr_auc": float(average_precision_score(y_te, probs)),
    "roc_auc": float(roc_auc_score(y_te, probs)),
}

print()
print(f"Best validation F1 during training : {best_val_f1_tuned:.4f}")
print(f"Best validation threshold seen     : {best_val_threshold_tuned:.2f}")
print()
print(f"Selected validation threshold (max F1): {SELECTED_THRESHOLD:.2f}")
print(f"Validation F1 at selected threshold   : {validation_threshold_metrics['f1']:.4f}")
print(f"Validation F2 at selected threshold   : {validation_threshold_metrics['f2']:.4f}")
print()
print(f"Test metrics (threshold = {SELECTED_THRESHOLD:.2f})")
for metric_name, metric_value in test_metrics.items():
    print(f"  {metric_name:<9}: {metric_value:.4f}")

test_predictions = full.iloc[test_idx][["datetime", "ACTUAL_POOL_PRICE"]].copy()
test_predictions["y_true"] = y_te.astype(int)
test_predictions["y_prob"] = probs
test_predictions["y_pred"] = preds



Best validation F1 during training : 0.5921
Best validation threshold seen     : 0.91

Selected validation threshold (max F1): 0.91
Validation F1 at selected threshold   : 0.5921
Validation F2 at selected threshold   : 0.6010

Test metrics (threshold = 0.91)
  f1       : 0.3630
  f2       : 0.4499
  precision: 0.2746
  recall   : 0.5354
  pr_auc   : 0.3820
  roc_auc  : 0.9339


In [10]:
# 10. Approximate top-20 feature importance using validation permutation importance
# If shuffling a feature hurts validation F1, that feature is helping the model.
# If shuffling a feature improves validation F1, that feature is probably noisy or harmful.

baseline_val_preds = (val_probs >= SELECTED_THRESHOLD).astype(int)
baseline_val_f1 = f1_score(y_va, baseline_val_preds, zero_division=0)
baseline_val_f2 = fbeta_score(y_va, baseline_val_preds, beta=2, zero_division=0)

rng = np.random.default_rng(RANDOM_SEED)
importance_rows = []
n_repeats = 3

for feature_index, feature_name in enumerate(FEATURES):
    repeat_f1_scores = []
    repeat_f2_scores = []
    for _ in range(n_repeats):
        X_perm = X_va.copy()
        X_perm[:, feature_index] = X_perm[rng.permutation(len(X_perm)), feature_index]

        perm_loader = DataLoader(
            TensorDataset(torch.tensor(X_perm), torch.tensor(y_va)),
            batch_size=BEST["batch_size"] * 2,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY,
        )

        perm_probs = []
        with torch.no_grad():
            for X_batch, _ in perm_loader:
                perm_probs.append(torch.sigmoid(final_model(X_batch.to(DEVICE))).cpu().numpy())
        perm_probs = np.concatenate(perm_probs)
        perm_preds = (perm_probs >= SELECTED_THRESHOLD).astype(int)
        repeat_f1_scores.append(f1_score(y_va, perm_preds, zero_division=0))
        repeat_f2_scores.append(fbeta_score(y_va, perm_preds, beta=2, zero_division=0))

    mean_perm_f1 = float(np.mean(repeat_f1_scores))
    mean_perm_f2 = float(np.mean(repeat_f2_scores))
    signed_change = float(baseline_val_f1 - mean_perm_f1)

    importance_rows.append(
        {
            "feature": feature_name,
            "baseline_f1": float(baseline_val_f1),
            "permuted_f1": mean_perm_f1,
            "baseline_f2": float(baseline_val_f2),
            "permuted_f2": mean_perm_f2,
            "signed_importance_f1_change": signed_change,
            "absolute_importance_f1_change": float(abs(signed_change)),
            "impact_sign": "+" if signed_change > 0 else "-" if signed_change < 0 else "0",
            "impact_direction": "helps" if signed_change > 0 else "hurts" if signed_change < 0 else "neutral",
        }
    )

feature_importance_df = pd.DataFrame(importance_rows).sort_values(
    "absolute_importance_f1_change", ascending=False
).reset_index(drop=True)

top_20_feature_importance = feature_importance_df.head(20)
print("Top 20 approximate feature importances (ordered by absolute F1 impact):")
top_20_feature_importance


Top 20 approximate feature importances (ordered by absolute F1 impact):


,feature,baseline_f1,permuted_f1,baseline_f2,permuted_f2,signed_importance_f1_change,absolute_importance_f1_change,impact_sign,impact_direction
0,ACTUAL_POOL_PRICE,0.592105,0.438250,0.601002,0.401328,0.153856,0.153856,+,helps
1,net_load_3h_change,0.592105,0.529698,0.601002,0.519671,0.062407,0.062407,+,helps
2,solar_total,0.592105,0.548985,0.601002,0.563352,0.043120,0.043120,+,helps
3,sin_hour_2,0.592105,0.551973,0.601002,0.558999,0.040132,0.040132,+,helps
4,wind_total,0.592105,0.561979,0.601002,0.518435,0.030126,0.030126,+,helps
5,ACTUAL_AIL,0.592105,0.573419,0.601002,0.562450,0.018686,0.018686,+,helps
6,renewable_generation,0.592105,0.578611,0.601002,0.548837,0.013494,0.013494,+,helps
7,dispatchable_ratio,0.592105,0.579801,0.601002,0.553920,0.012304,0.012304,+,helps
8,cos_hour_2,0.592105,0.580059,0.601002,0.594672,0.012046,0.012046,+,helps
9,hydro_total,0.592105,0.580170,0.601002,0.574356,0.011935,0.011935,+,helps


In [11]:
# 11. Save outputs
training_history_df = pd.DataFrame(training_history)

joblib.dump(scaler_final, OUTPUT_DIR / "scaler.joblib")
torch.save(final_model.state_dict(), OUTPUT_DIR / "best_model.pt")

latest_checkpoint = CHECKPOINT_DIR / "latest_model.pt"
if latest_checkpoint.exists():
    (OUTPUT_DIR / "latest_model.pt").write_bytes(latest_checkpoint.read_bytes())

pd.DataFrame(
    {
        "cv_fold_f1": fold_f1s,
        "cv_fold_f2_at_best_f1_threshold": fold_f2s_at_best_f1_threshold,
        "cv_best_threshold": fold_thresholds,
    }
).to_csv(OUTPUT_DIR / "cv_results.csv", index=False)
search_results_df.to_csv(OUTPUT_DIR / "search_results.csv", index=False)
threshold_search_df.to_csv(OUTPUT_DIR / "validation_threshold_search.csv", index=False)
feature_importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
top_20_feature_importance.to_csv(OUTPUT_DIR / "top_20_feature_importance.csv", index=False)
training_history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
test_predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "best_params": BEST,
            "candidate_grid": CANDIDATE_GRID,
            "feature_count": len(FEATURES),
            "cv_fold_f1s": [round(float(x), 4) for x in fold_f1s],
            "cv_mean_f1": round(float(np.mean(fold_f1s)), 4),
            "cv_std_f1": round(float(np.std(fold_f1s)), 4),
            "cv_fold_f2s_at_best_f1_threshold": [round(float(x), 4) for x in fold_f2s_at_best_f1_threshold],
            "cv_mean_f2_at_best_f1_threshold": round(float(np.mean(fold_f2s_at_best_f1_threshold)), 4),
            "cv_std_f2_at_best_f1_threshold": round(float(np.std(fold_f2s_at_best_f1_threshold)), 4),
            "cv_best_thresholds": [round(float(x), 2) for x in fold_thresholds],
            "best_validation_f1_with_tuned_threshold": round(float(best_val_f1_tuned), 4),
            "best_validation_f2_with_tuned_threshold": round(float(best_val_f2_tuned), 4),
            "best_validation_threshold_during_training": round(float(best_val_threshold_tuned), 4),
            "threshold_selection_metric": "f1",
            "selected_threshold": round(float(SELECTED_THRESHOLD), 4),
            "validation_threshold_metrics": {
                key: round(float(value), 4) for key, value in validation_threshold_metrics.items()
            },
            "test": {
                key: round(float(value), 4) for key, value in test_metrics.items()
            },
        },
        f,
        indent=2,
    )

print()
print(f"Outputs saved to: {OUTPUT_DIR}")



Outputs saved to: random_search_run
